# Lab 17: A2C on CartPole

**Module 17** | Read `notes/17-policy-gradients-actor-critic.pdf` before running this notebook.

Train an A2C actor-critic agent on CartPole-v1 and compare learning speed to the DQN lab.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


# Lab 17: A2C on CartPole

**Module 17** | Read `notes/17-policy-gradients-actor-critic.pdf` before running this notebook.

Train an A2C actor-critic agent on CartPole-v1 and compare learning speed to the DQN lab.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Actor-Critic (A2C) on CartPole

**A2C** (Advantage Actor-Critic) is an on-policy algorithm that learns both a policy (actor) and a value function (critic).
Unlike DQN, it optimizes the policy directly with policy gradients rather than estimating Q-values for every action.

Advantages of actor-critic methods on CartPole:

- Often learn smoothly on continuous control-style tasks.
- No replay buffer; updates use the current policy's rollouts.
- The critic reduces variance of policy-gradient estimates.

You will train A2C for **30,000 timesteps** on the same CartPole-v1 task used in Lab 16, then compare final performance.


In [ ]:
import gymnasium as gym
import numpy as np
from pathlib import Path
from stable_baselines3 import A2C
from stable_baselines3.common.monitor import Monitor

ROOT = Path("..").resolve()

env = Monitor(gym.make("CartPole-v1"))
model = A2C("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=30_000)

rewards = env.get_episode_rewards()
mean_last_10 = float(np.mean(rewards[-10:])) if len(rewards) >= 10 else float(np.mean(rewards))

print(f"Episodes completed: {len(rewards)}")
print(f"Mean reward (last 10 episodes): {mean_last_10:.2f}")
print(f"Max episode reward: {max(rewards)}")
print(f"Overall mean reward: {np.mean(rewards):.2f}")

env.close()


## Comparison with DQN (Lab 16)

Both DQN and A2C typically reach **similar final performance** on CartPole-v1 when trained for 30,000 timesteps.
Mean reward over the last 10 episodes is often in the **400 to 500** range once training converges.

| Aspect | DQN (Lab 16) | A2C (this lab) |
|--------|--------------|----------------|
| Learning style | Off-policy (replay buffer) | On-policy (current rollouts) |
| What is learned | Q-function → greedy policy | Policy + value function jointly |
| Typical CartPole outcome | Similar mean reward | Similar mean reward |
| Sample efficiency | Can reuse old transitions | Discards data after each update |

**Practical note:** CartPole is easy enough that both families succeed; the difference shows more on harder environments (continuous action spaces, sparse rewards, long horizons).
On this task, compare your printed **mean reward (last 10 episodes)**, values within ~10% of Lab 16 are expected.
